# Kizagan-E4B Türkçe FC — Benchmark & Forgetting Testi

Model: `Tuguberk/Kizagan-E4B-Turkish-Agent-FunctionCalling-Hermes`

İki bölüm:

**BÖLÜM A — Catastrophic Forgetting Testi**
Alican'ın base modelinin değerlendirdiği iki testi senin model üzerinde çalıştırır. Amaç: function-calling fine-tune'u genel Türkçe/muhakeme yeteneğini bozdu mu?

| Test | Few-shot | Alican Kizagan (referans) | Base gemma-4-E4B-it (referans) |
|------|----------|---------------------------|-------------------------------|
| mmlu_tr-v0.2 | 5 | %33.18 | %30.42 |
| gsm8k_tr-v0.2 | 5 | %35.94 | %14.06 |

> Referans skorlar Alican'ın model kartından. Bu modelleri YENİDEN test etmiyoruz — sadece senin modelini ölçüp bu değerlerle kıyaslıyoruz.

**BÖLÜM B — Function Calling Testi (held-out)**
`atasoglu/turkish-function-calling-20k` ile out-of-distribution function calling değerlendirmesi. Bu dataset eğitimde kullanılmadı.

In [ ]:
# ============================================================
# A1 — GPU
# ============================================================
import subprocess, torch
print(subprocess.run(['nvidia-smi'], capture_output=True, text=True).stdout)
print(f'GPU: {torch.cuda.get_device_name(0)} | VRAM: {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB')

## Kurulum (Bölüm A için)

malhajar17 Türkçe fork'u — `mmlu_tr-v0.2` ve `gsm8k_tr-v0.2` görev tanımları burada.

In [ ]:
# ============================================================
# A2 — lm-eval Türkçe fork kurulumu
# ============================================================
import os
if not os.path.exists('lm-evaluation-harness_turkish'):
    !git clone https://github.com/malhajar17/lm-evaluation-harness_turkish
%cd lm-evaluation-harness_turkish
!pip install -e . -q
!pip install -U transformers accelerate sentencepiece -q
%cd ..
print('✓ Kurulum tamam. Sürüm çakışması uyarısı görürsen RESTART + bu hücreyi atla.')

## Model Yüklenebilirlik Testi

In [ ]:
# ============================================================
# A3 — Model erişim testi
# ============================================================
from huggingface_hub import login
from getpass import getpass
try:
    login(token=getpass('HF Token (gated base için, Enter=atla): ') or None)
except Exception: pass

MODEL_ID = "Tuguberk/Kizagan-E4B-Turkish-Agent-FunctionCalling-Hermes"

from transformers import AutoConfig, AutoTokenizer, AutoModelForCausalLM
import torch
cfg = AutoConfig.from_pretrained(MODEL_ID)
print(f'✓ Config — model_type: {cfg.model_type}')
_t = AutoTokenizer.from_pretrained(MODEL_ID)
print(f'✓ Tokenizer — vocab: {len(_t)}')
_m = AutoModelForCausalLM.from_pretrained(MODEL_ID, torch_dtype=torch.bfloat16, device_map='auto')
print(f'✓ Model — {sum(p.numel() for p in _m.parameters())/1e9:.2f}B param')
del _m; torch.cuda.empty_cache()

## Forgetting Testi Ayarları

`LIMIT=20` ile başla (hızlı doğrulama). Gerçek skor için `LIMIT=None` (MMLU saatlerce sürer).

In [ ]:
# ============================================================
# A4 — Ayarlar
# ============================================================
MODEL_ID = "Tuguberk/Kizagan-E4B-Turkish-Agent-FunctionCalling-Hermes"
OUTPUT_DIR = './forgetting_results'
LIMIT = 20          # TEST. Gerçek skor için None.
DTYPE = 'bfloat16'

# Alican'ın kullandığı 2 test, leaderboard few-shot ayarlarıyla
TASKS = {'mmlu_tr-v0.2': 5, 'gsm8k_tr-v0.2': 5}

import os; os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f'Model: {MODEL_ID}\nLIMIT: {LIMIT}\nTestler: {list(TASKS)}')

In [ ]:
# ============================================================
# A5 — Forgetting testlerini çalıştır
# ============================================================
import subprocess, time
_lim = f'--limit {LIMIT}' if LIMIT else ''
for task, nshot in TASKS.items():
    print(f'\n{"="*55}\n{task} (few-shot={nshot})\n{"="*55}')
    t0 = time.time()
    out = f'{OUTPUT_DIR}/{task.replace("/","_")}'
    cmd = (f'lm_eval --model hf '
           f'--model_args pretrained={MODEL_ID},dtype={DTYPE},trust_remote_code=True '
           f'--tasks {task} --num_fewshot {nshot} --batch_size auto '
           f'--output_path {out} {_lim}')
    p = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    print(p.stdout[-2500:])
    if p.returncode != 0:
        print('⚠ HATA:', p.stderr[-1500:])
    print(f'Süre: {(time.time()-t0)/60:.1f} dk')

In [ ]:
# ============================================================
# A6 — Forgetting sonuçları + referansla kıyas
# ============================================================
import json, glob

REF = {  # Alican'ın model kartından
    'mmlu_tr-v0.2':  {'kizagan': 33.18, 'gemma_base': 30.42, 'metric': 'acc,none'},
    'gsm8k_tr-v0.2': {'kizagan': 35.94, 'gemma_base': 14.06, 'metric': 'exact_match,strict-match'},
}

print(f'{"TEST":<18}{"SENİN":>8}{"KIZAGAN":>10}{"GEMMA":>8}{"Δ(Kizagan)":>12}')
print('-'*56)
for task in TASKS:
    files = glob.glob(f'{OUTPUT_DIR}/{task.replace("/","_")}/**/results_*.json', recursive=True)
    if not files:
        print(f'{task:<18}{"YOK":>8}'); continue
    res = json.load(open(sorted(files)[-1]))['results'].get(task, {})
    mkey = REF[task]['metric']
    val = res.get(mkey)
    if val is None:  # esnek arama
        for k,v in res.items():
            if REF[task]['metric'].split(',')[0] in k and isinstance(v,(int,float)):
                val = v; break
    if isinstance(val,(int,float)):
        s = val*100
        kz = REF[task]['kizagan']
        print(f'{task:<18}{s:>7.2f}%{kz:>9.2f}%{REF[task]["gemma_base"]:>7.2f}%{s-kz:>+11.2f}')
    else:
        print(f'{task:<18}{"?":>8}')

print('''
YORUM:
  Δ(Kizagan) ~0 veya pozitif  → forgetting YOK, yetenek korundu ✓
  Δ(Kizagan) hafif negatif    → kabul edilebilir, FC eğitimi normal etki
  Δ(Kizagan) belirgin negatif → catastrophic forgetting, müdahale gerek
  Not: LIMIT!=None ise sonuçlar temsili değil.''')

---

# BÖLÜM B — Function Calling Testi (atasoglu, held-out)

`atasoglu/turkish-function-calling-20k` ile out-of-distribution test. Bu veri eğitimde kullanılmadı, farklı kaynak + farklı format. Semantik eşleşme (fonksiyon adı + argüman anahtarları) ölçülür — string exact-match değil.

In [ ]:
# ============================================================
# B1 — Model + tokenizer yükle (FC testi için, doğru template ile)
# ============================================================
import torch, json, re, random
from transformers import AutoModelForCausalLM, AutoTokenizer

MODEL_ID = "Tuguberk/Kizagan-E4B-Turkish-Agent-FunctionCalling-Hermes"
model = AutoModelForCausalLM.from_pretrained(MODEL_ID, torch_dtype=torch.bfloat16, device_map='auto')
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model.eval()

# chat_template'in tool injection içerdiğini doğrula
assert 'tools' in (tokenizer.chat_template or '').lower(), \
    'UYARI: tokenizer chat_template tool injection içermiyor olabilir!'
print('✓ Model + tokenizer yüklendi, template tool destekli')

In [ ]:
# ============================================================
# B2 — atasoglu datasetini yükle ve örnekle
# ============================================================
from datasets import load_dataset

N_SAMPLES = 200   # held-out test örneği (tamamı 22k, alt küme yeterli)
ds = load_dataset('atasoglu/turkish-function-calling-20k', split='train')
random.seed(42)
idxs = random.sample(range(len(ds)), min(N_SAMPLES, len(ds)))
samples = ds.select(idxs)
print(f'Toplam {len(ds)} → {len(samples)} örnek seçildi')
print('\nÖrnek:')
print('  query   :', samples[0]['query'][:100])
print('  tools   :', samples[0]['tools'][:120])
print('  answers :', samples[0]['answers'][:120])

In [ ]:
# ============================================================
# B3 — Generate fonksiyonu (greedy, doğru template)
# ============================================================
def generate(query, tools_json, max_new_tokens=256):
    tools = json.loads(tools_json) if isinstance(tools_json, str) else tools_json
    msgs = [{'role': 'user', 'content': query}]
    enc = tokenizer.apply_chat_template(
        msgs, tools=tools, tokenize=True,
        add_generation_prompt=True, return_tensors='pt', return_dict=True
    ).to(model.device)
    with torch.no_grad():
        out = model.generate(input_ids=enc['input_ids'],
                             attention_mask=enc['attention_mask'],
                             max_new_tokens=max_new_tokens, do_sample=False,
                             pad_token_id=tokenizer.eos_token_id)
    resp = tokenizer.decode(out[0][enc['input_ids'].shape[-1]:], skip_special_tokens=False)
    return resp.split('<turn|>')[0].strip()

# Test
print(generate(samples[0]['query'], samples[0]['tools'])[:300])

In [ ]:
# ============================================================
# B4 — FC değerlendirme: semantik eşleşme
# ============================================================
_TC = re.compile(r'<tool_call>\s*(.*?)\s*</tool_call>', re.DOTALL)

def extract_calls(text):
    "Model çıktısından {name, arg_keys} çıkar."
    calls = []
    for m in _TC.findall(text):
        try:
            j = json.loads(m.strip())
            calls.append((j.get('name',''), set((j.get('arguments') or {}).keys())))
        except Exception: pass
    return calls

def gold_calls(answers_json):
    "atasoglu answers'tan {name, arg_keys} çıkar."
    try:
        ans = json.loads(answers_json) if isinstance(answers_json,str) else answers_json
    except Exception: return []
    if ans is None: return []
    out = []
    for a in ans:
        fn = a.get('function', {})
        name = fn.get('name','')
        args = fn.get('arguments','{}')
        try:
            keys = set(json.loads(args).keys()) if isinstance(args,str) else set((args or {}).keys())
        except Exception: keys = set()
        out.append((name, keys))
    return out

n_total = n_produced = n_name_ok = n_count_ok = n_args_ok = 0
samples_log = []

for ex in samples:
    gold = gold_calls(ex['answers'])
    if not gold:     # answers boş/null → atla
        continue
    n_total += 1
    resp = generate(ex['query'], ex['tools'])
    pred = extract_calls(resp)

    if pred:
        n_produced += 1
    gold_names = {g[0] for g in gold}
    pred_names = {p[0] for p in pred}
    # isim eşleşmesi: en az bir doğru isim
    if pred_names & gold_names:
        n_name_ok += 1
    # sayı eşleşmesi
    if len(pred) == len(gold):
        n_count_ok += 1
    # argüman anahtarı eşleşmesi (ilk eşleşen isim için)
    arg_match = False
    for pn, pk in pred:
        for gn, gk in gold:
            if pn == gn and pk == gk:
                arg_match = True
    if arg_match:
        n_args_ok += 1

    if len(samples_log) < 5:
        samples_log.append((ex['query'][:80], gold, pred))

print(f'{"-"*52}')
print(f'{"METRİK":<38}{"ORAN":>12}')
print(f'{"-"*52}')
print(f'{"tool_call üretti":<38}{100*n_produced/n_total:>11.1f}%')
print(f'{"Fonksiyon adı doğru (≥1)":<38}{100*n_name_ok/n_total:>11.1f}%')
print(f'{"Çağrı sayısı doğru":<38}{100*n_count_ok/n_total:>11.1f}%')
print(f'{"Argüman anahtarları eşleşti":<38}{100*n_args_ok/n_total:>11.1f}%')
print(f'{"-"*52}\nDeğerlendirilen: {n_total}')

print('\n=== Örnekler ===')
for q,g,p in samples_log:
    print(f'\nSoru : {q}')
    print(f'Gold : {[(n,sorted(k)) for n,k in g]}')
    print(f'Pred : {[(n,sorted(k)) for n,k in p]}')

## Yorum

- **Fonksiyon adı doğruluğu** en kritik metrik — model doğru aracı seçiyor mu.
- **Argüman anahtarı eşleşmesi** daha zorlu — bu held-out, farklı kaynak veri.
- Skorlar kendi Hermes benchmark'ından (in-distribution, ~%84) düşük çıkabilir; bu normal: atasoglu farklı dağılım + çok teknik Python-utility tool'lar (domain shift). Düşüş forgetting değil, genelleme sınırıdır.
- Yine de fonksiyon adı doğruluğu yüksekse, model format ve niyet eşlemeyi gerçekten öğrenmiş demektir.